### Jupyter Notebook which plots the optimized QAOA angles. This Notebook only serves debugging

In [1]:
%load_ext autoreload
%autoreload 2
# load the qaoa parameters which were previously calculated
import numpy as np
import matplotlib.pyplot as plt
from qaoa_pipeline import *
import pandas as pd
import os
import re
import time
import pickle
from pathlib import Path
from datetime import datetime
from collections import defaultdict


folder = 'local/ssh_qaoa_params_v1/'
files = os.listdir(folder)

output_folder = 'local/parameter_plots/'
os.makedirs(output_folder, exist_ok=True)

# Example filename:
# candidate_space_8_candidates_50_busses_0.6_load_factor_2026-04-15_01-45-21_2_p0_128.pkl

filename_pattern = re.compile(
    r'^candidate_space_(?P<candidate_space>\d+)_candidates_50_busses_'
    r'(?P<load>\d+\.\d+)_load_factor_'
    r'(?P<timestamp>\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})_'
    r'(?P<instance>\d+)_p0_(?P<p>\d+)\.pkl$'
)

def parse_filename(filename):
    match = filename_pattern.match(filename)
    if not match:
        return None  # instead of raising
    return {
        "candidate_space": int(match.group("candidate_space")),
        "load": float(match.group("load")),
        "timestamp": datetime.strptime(match.group("timestamp"), "%Y-%m-%d_%H-%M-%S"),
        "instance": int(match.group("instance")),
        "p": int(match.group("p")),
    }

# Filter out files that don't match before sorting
files = [f for f in os.listdir(folder) if parse_filename(f) is not None]

# hierarchical sort by candidate space, load factor, instance, p
def sort_key(filename):
    parsed = parse_filename(filename)
    return (
        parsed["candidate_space"],
        parsed["load"],
       # parsed["instance"],
        parsed["p"],
    )

files_sorted = sorted(files, key=sort_key)

for file in files_sorted:
    print(file)

# group files by (candidate_space, load)
groups = defaultdict(list)

for file in files:
    parsed = parse_filename(file)
    groups[(parsed["candidate_space"], parsed["load"])].append(file)

# optional: sort files within each group too
for key in groups:
    groups[key] = sorted(groups[key], key=sort_key)

candidate_space_2_candidates_50_busses_0.1_load_factor_2026-04-15_01-45-21_4_p0_64.pkl
candidate_space_2_candidates_50_busses_0.1_load_factor_2026-04-15_01-45-21_5_p0_128.pkl
candidate_space_2_candidates_50_busses_0.1_load_factor_2026-04-15_01-45-21_3_p0_256.pkl
candidate_space_2_candidates_50_busses_0.2_load_factor_2026-04-15_01-45-21_2_p0_64.pkl
candidate_space_2_candidates_50_busses_0.2_load_factor_2026-04-15_01-45-21_7_p0_128.pkl
candidate_space_2_candidates_50_busses_0.2_load_factor_2026-04-15_01-45-21_2_p0_256.pkl
candidate_space_2_candidates_50_busses_0.3_load_factor_2026-04-15_01-45-21_0_p0_64.pkl
candidate_space_2_candidates_50_busses_0.3_load_factor_2026-04-15_01-45-21_8_p0_128.pkl
candidate_space_2_candidates_50_busses_0.3_load_factor_2026-04-15_01-45-21_4_p0_256.pkl
candidate_space_2_candidates_50_busses_0.4_load_factor_2026-04-15_01-45-21_7_p0_64.pkl
candidate_space_2_candidates_50_busses_0.4_load_factor_2026-04-15_01-45-21_1_p0_128.pkl
candidate_space_2_candidates_50_buss

In [ ]:
for file in files_sorted:
    with open(folder+file, "rb") as f:
        data = pickle.load(f)
        AR = data["AR"]
        gammas = data["gammas"]
        betas = data["betas"]
        runtime = data["runtimes"]
        p0 = data["p0"]
        num_generators = data["num_generators"]
        load = data["load"]
        chosen_file = data["source_file"]
        
        plt.figure()
        plt.plot(range(len(gammas)), gammas, label=r'$\gamma$')
        plt.plot(range(len(betas)), betas, label=r'$\beta$')
        title = (
            str(chosen_file) + '\n' +
            f'AR={list(AR.values())[-1]}, runtime={runtime}, p0={p0}, '
            'delta_p = 5, p_max = 3/2*p0, C=p_max//10,\n'
            'epsilon=1, tau=3, stepsize=0.001, optsteps=100')
        
        plt.title(title)
        plt.xlabel('layer index')
        plt.legend()
        safe_name = file.replace(".pkl", ".png")
        save_path = os.path.join(output_folder, safe_name)
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()  # prevents memory issues